In [5]:
import maos_utils as mu
import numpy as np
from pathlib import Path
import astropy.units as u
from astropy.io import fits
from PyAstronomy import pyasl

### **Functions**

In [3]:
def SNR_KAON113(S, ti, N, ND, NS, NR):
    """
    Inputs:
    -------
    S   : float
        Number of photons per sec sensed by subaperture

    ti  : float
        Integration time in sec

    N   : float
        Number of pixels per subaperture
    
    ND  : float
        Dark noise in events per pixel per second

    NS  : float
        Total sky background noise (1.67 photons per second for LBWFS)

    NR  : float
        Read noise

    Outputs:
    --------
    SNR : float
       Signal-to-noise ratio

    By Brooke DiGia
    """
    SNR = S * np.sqrt( ti / ( S + NS + N*(ND + ( NR**2.0 / ti )) ) )
    return SNR

In [4]:
# Spot size calculation for SHWFS LGS-AO mode (KAON 1317)
np.sqrt(1.5**2.0 + (699*1e-3)**2.0 + mu.seeing_limit_spot_size(589.0*1e-9, 0.20)**2.0 + (0.27*3)**2.0)

1.9354964816326197

### **Expected power from LGS**

Science image FITS headers have ``LSAMPPWR`` keyword to indicate the laser guide star power (Watts). Keyword definition does not specify if this is upon launch or after downward propagation through atmosphere to telescope, but it is highly likely it is the setting at launch. Analysis below assumes this is laser power at launch (prior to upward propagation through atmosphere).

In [9]:
# Use already-run base GC science image (20240823) since it is closest in date to the new RTC telemetry science image (this file is lacking the LSAMPPWR keyword)
file = "/u/bdigia/work/ao/airopa_input/20170823nirc2_kp/c2061_psf.fits"
with fits.open(file) as fits_file:
    hdu = fits_file[0]
    psf = hdu.data
    hdr = hdu.header
    lgs_pwr_20240807 = hdr['LSAMPPWR']
print(f"Laser power for 20240807 new RTC telemetry science image = {lgs_pwr_20240807} Watts")


Laser power for 20240807 new RTC telemetry science image = 22.487801 Watts


### **LBWFS**

#### *Spot Size*

In [5]:
# Select a date of observation for which to perform validation
dates = ['20150809']
skyroot = Path("/Users/bdigia/work/ao/airopa_input/")

names = []
datecol = []
sky_paths = []
for date in dates:
    name = [f.as_posix()[-14:-9] for f in skyroot.glob(f"{date}nirc2_kp/*_psf.fits")]
    paths = [f.as_posix() for f in skyroot.glob(f"{date}nirc2_kp/*_psf.fits")]
    names.extend(name)
    temp = np.full(len(name), date)
    datecol.extend(temp.tolist())
    sky_paths.extend(paths)
names = np.array(names)
datecol = np.array(datecol)
namesanddates = np.column_stack((names, datecol))
print(namesanddates)

[['c0199' '20150809']
 ['c0207' '20150809']
 ['c0208' '20150809']
 ['c0210' '20150809']
 ['c0256' '20150809']
 ['c0257' '20150809']
 ['c0258' '20150809']]


In [6]:
# Pull atm/weather info for sky file(s)
for i, sky in enumerate(namesanddates):
    sky_file = skyroot.as_posix() + f"/{sky[1]}nirc2_kp/{sky[0]}_psf.fits"
    sky_folder = skyroot.as_posix() + f"/{sky[1]}nirc2_kp/"
    fried, turbpro, windspds, winddrcts, dimm, mass, dimmtime, masstime, tau_0, theta_0, sigma_DM = mu.estimate_on_sky_conditions(sky_file, 
                                                                                                                                  sky_folder, 
                                                                                                                                  verbose=True)
    print(fried)

NOTE: Results for MAOS configuration files marked with ***

20150809.dimm.dat exists in directory /Users/bdigia/work/ao/airopa_input/20150809nirc2_kp/, not downloading.
20150809.masspro.dat exists in directory /Users/bdigia/work/ao/airopa_input/20150809nirc2_kp/, not downloading.
cfht-wx.2015.dat exists in directory /Users/bdigia/work/ao/airopa_input/20150809nirc2_kp/, not downloading.
phto.20150809.dat exists in directory /Users/bdigia/work/ao/airopa_input/20150809nirc2_kp/, not downloading.

Date of observation is  2015-08-09 (UT)
Exposure time is	06:29:33.663 to 06:30:16.138 (UT)

Closest DIMM data to beginning of exposure:	0.8200	at 2015:8:9:6:30:57
Closest DIMM data to end of exposure:		0.8200	at 2015:8:9:6:30:57
Average DIMM over exposure:			0.8200

Closest MASS data to beginning of exposure:  [2.53e-19 1.45e-14 3.13e-22 2.94e-16 1.19e-21 1.13e-14 1.40e-01] at  2015:8:9:6:30:27
Closest MASS data to end of exposure:	     [2.53e-19 1.45e-14 3.13e-22 2.94e-16 1.19e-21 1.13e-14 1.40e

In [7]:
print(fried)

0.18640226934922785 m


In [8]:
# Calculate seeing-limited disk for each wavefront sensor for calculated r0 based on atmospheric
# data from date's night of observation
band_wvl = 0.641e-6*u.m
lbwfs_seeing_limit_spot_size = mu.seeing_limit_spot_size(band_wvl, fried)
print(f"Low Bandwith WFS seeing limited spot size = {lbwfs_seeing_limit_spot_size:.3f} arcsec")

Low Bandwith WFS seeing limited spot size = 0.692 arcsec


In [9]:
# Gaussian convolve seeing-limited disk with intrinsic 0.5 Hartmann spot size FWHM (KAON 245)
diffrac_spot_lbwfs = ((0.641e-6) / 0.563)*1.22*206265*2
theta_beta_lbwfs = np.sqrt(lbwfs_seeing_limit_spot_size**2.0 + 0.5**2.0 + diffrac_spot_lbwfs**2.0)
print(f"Low Bandwith WFS theta_beta spot size = {theta_beta_lbwfs:.3f} arcsec")

Low Bandwith WFS theta_beta spot size = 1.028 arcsec


In [10]:
((0.641e-6) / 0.563)*1.22*206265*2

0.5730136955595027

#### *Sky Background and Flux*

In [11]:
# LBWFS parameters
band = "R"
side = 0.563
throughput = 0.03
ps = 0.148
time = 15
m = 14
pix_per_ap = 16.7*16.7
sigma_e = 7.96
field_stop = 2.44 # arcsec diameter

In [12]:
# Fixed parameters
c = 2.99792458e8   # Speed of light (m s^-1)
h = 6.6260755e-27  # Plank constant (erg s)
# Bands' names, effective wavelengths (microns), equivalent widths
# (microns), fluxes (10^-11 erg s^-1 cm^-2 A^-1) and background
# in (magnitudes arcsec^-2) [1, 2, 3].
bands = {'name': ["U", "B", "V", "R", "I", "J", "H", "K"],
         'lambd': [0.366, 0.438, 0.545, 0.641, 0.798, 1.22, 1.63, 2.19],
         'delta_lambd': [0.0665, 0.1037, 0.0909, 0.1479, 0.1042, 0.3268, 
                         0.2607, 0.5569],
         'phi_erg': [417.5, 632, 363.1, 217.7, 112.6, 31.47, 11.38, 3.961],
         'bkg_m': [21.6, 22.3, 21.1, 20.3, 19.2, 14.8, 13.4, 12.6]}
# Get band's data
band_idx = np.where(np.array(bands['name']) == band)[0][0]
# Band effective wavelength (microns)    
lambd = float(bands['lambd'][band_idx])
# Band equivalent width (microns)
delta_lamb = float(bands['delta_lambd'][band_idx])
# Flux (erg s^-1 cm^-2 A^-1)
phi_erg = float(bands['phi_erg'][band_idx])
print(phi_erg*1.0e-11)
# Background magnitude (arcsec^-2)    
bkg_m = float(bands['bkg_m'][band_idx])
# Band frequency (Hz)
f = c / (lambd * 1.0e-6)
# Numeric flux (s^-1 cm^-2 A^-1)
phi_n = (phi_erg * 1.0e-11) / ( h * f )
print(phi_n)
# Zeropoint (m = 0) number of photons on detector
n_ph_0 = phi_n * ( (side * 1e2) ** 2) * time * delta_lamb * 1e4 * throughput
# Number of star photons
n_ph_star = n_ph_0 * ( 10**(-0.4 * (m)) ) 
print(f"Number of star photons on LBWFS at integration time = {n_ph_star:.3f} photons/subaperture")
# Number of background photons (px^-1)
n_ph_bkg = n_ph_0 * ( 10**(-0.4 * bkg_m) ) * (ps)**2.0
print(f"Number of background photons on LBWFS at integration time = {n_ph_bkg} bkgrnd photons/px, {n_ph_bkg/time} photons/s/px")

2.177e-09
702.4887530981614
Number of star photons on LBWFS at integration time = 3722.518 photons/subaperture
Number of background photons on LBWFS at integration time = 0.2462409532912565 bkgrnd photons/px, 0.016416063552750434 photons/s/px


In [13]:
# Convert magnitude to flux density based on Bessel et al. 1998
mag_R = 0.0
phi_erg, le = pyasl.magToFluxDensity_bessel98("R", mag_R, "lam")
print(f"R-band flux density for mag {mag_R} star = {phi_erg} [erg s^-1 cm^-2 A^-1]")

R-band flux density for mag 0.0 star = 2.1777097723531546e-09 [erg s^-1 cm^-2 A^-1]


In [14]:
# KAON 245 gives incident flux for mag. M star at a subaperture empirical relation:
# 1.1 * 10^(8 - (M/2.5))
M = 14
lbwfs_flux_kaon245 = 1.1 * 10**(8 - (M/2.5))
print(f"Number of star photons on LBWFS = {lbwfs_flux_kaon245:.3f} photons/sec/subaperture")
print(f"Multiplying by integration time to get just photons/subaperture to match n_photons --> {lbwfs_flux_kaon245*time} photons/subaperture")

Number of star photons on LBWFS = 276.308 photons/sec/subaperture
Multiplying by integration time to get just photons/subaperture to match n_photons --> 4144.612611990811 photons/subaperture


In [15]:
# KAON 245 gives sky background NOISE 1.67 e/sec/arcsec from 2.8 photons/0.785 arcsec^2. 0.148 arcsec/px scale for LBWFS
lbwfs_bkgrnd_KAON245 = (2.8/0.785) * 0.148 * 0.148
print(f"KAON 245 gives LBWFS sky background as {lbwfs_bkgrnd_KAON245} e/sec/pixel. At integration time this becomes {lbwfs_bkgrnd_KAON245*time} e/pixel")
print(f"KAON 332 gives dark sky background as 0.4 e-/s/pix, {0.4*15.0} e-/pix at integration time")

KAON 245 gives LBWFS sky background as 0.07812891719745221 e/sec/pixel. At integration time this becomes 1.1719337579617832 e/pixel
KAON 332 gives dark sky background as 0.4 e-/s/pix, 6.0 e-/pix at integration time


In [16]:
# Photometric zero point R0 = 18.0 +/- 0.1 from KAON 1303 referencing KAON 332. KAON 332 gves 3 photon/second/subaperture for mag 19 star.
m = 0.0
counts_persec = 10**( (18.0 - m) / 2.5)
print(f"Absolute counts per second for LBWFS = {counts_persec} cts/sec. At integration time = {counts_persec*time:.3f}")

Absolute counts per second for LBWFS = 15848931.924611142 cts/sec. At integration time = 237733978.869


In [17]:
# LBWFS CCD center dark current from KAON 332: 1.12 +/- 0.1 e-/s/pix
n_dark = (1.12) * time * pix_per_ap
print(f"LBWFS dark current at integration time = {n_dark} e-/subaperture; noise sqrt(n_dark) = {np.sqrt(n_dark)}")

LBWFS dark current at integration time = 4685.352 e-/subaperture; noise sqrt(n_dark) = 68.44963111661012


In [18]:
SNR = n_ph_star / np.sqrt(n_ph_star + 2.0*pix_per_ap*n_ph_bkg + n_dark + pix_per_ap*sigma_e**2)
print(SNR)

22.990717133364367


In [19]:
wvl = 0.641e-6
d = 0.5625
r0 = fried.value
theta_gs = theta_beta_lbwfs * (1.0/0.148) * (24.0 * 1e-6) # spot size from arcsec to meters using KAON 245 24 micron pixel scale and size for LBWFS detector
ctrlloop = 0.393
( ( wvl / ( (8 / (3*np.pi)) * ( np.exp(-0.115 * (d/r0)) + np.exp(-0.155 * (d/r0)**2.0) ) ) ) + ( ( np.pi*d*0.0 ) / 8.0 ) ) * (ctrlloop/SNR) * (1e9) # from m to nm

13.579919368644617

In [20]:
# Compare to KAON 113's equation using keck_nea_photon outputs

# Zeropoint (m = 0) number of photons on detector
n_ph_0 = phi_n * ( (side * 1e2) ** 2) * delta_lamb * 1e4 * throughput # per second 
# Number of star photons
n_ph_star = n_ph_0 * ( 10**(-0.4 * m) ) 
time = 15
pix_per_ap = 16.7*16.7
# Number of background photons
n_ph_bkg = n_ph_0 * ( 10**(-0.4 * bkg_m) ) * ps**2.0 # per second
# Dark current, taken from KAON because currently not calculated in keck_nea_photons (KAON 332)
n_dark = 1.12 # e-/s/pix
# Readnoise taken from KAON 332, currently not calculated in keck_nea_photons
sigma_e = 5.82

SNR_KAON113(n_ph_star, time, pix_per_ap, n_dark, n_ph_bkg, sigma_e)

38496.066489210454

Very similar to the above SNR from keck_nea_photons using keck_nea_photons version of equation

In [21]:
# Compare to KAON 113's equation using KAON info

# Number of photons per sec per subaperture
M = 14
lbwfs_flux_kaon245 = 1.1 * 10**(8 - (M/2.5)) # e/s/subaperture
# Number of background photons
n_ph_bkg = n_ph_0 * ( 10**(-0.4 * bkg_m) ) * ps**2.0 # per second
n_ph_bkg = 0.4 #e-/s/pix
# Dark current, taken from KAON because currently not calculated in keck_nea_photons (KAON 332)
n_dark = 1.12 # e-/s/pix
# Readnoise taken from KAON 332, currently not calculated in keck_nea_photons
sigma_e = 5.82 #e-/pix

SNR_KAON113(n_ph_star, time, pix_per_ap, n_dark, n_ph_bkg, sigma_e)

38496.066414480105

### **STRAP**

Fast SHWFS for LGSWFS is not validated against ``keck_nea_photons()`` since the spot size is hard-coded to known 1.5 arcsec (then converted to radians
as for other WFS)

#### *Spot Size*

In [24]:
fried = 0.20
band_wvl = 0.641e-6
strap_seeing_limit_spot_size = mu.seeing_limit_spot_size(band_wvl, fried)
print(f"STRAP WFS seeing limited spot size = {strap_seeing_limit_spot_size:.3f} arcsec")
# Gaussian convolve seeing-limited disk with intrinsic 0.6 spot size (KAON 422 Figure 5)
theta_beta_strap = np.sqrt(strap_seeing_limit_spot_size**2.0 + 0.625**2.0)
print(f"STRAP WFS theta_beta spot size = {theta_beta_strap:.3f} arcsec")

STRAP WFS seeing limited spot size = 0.645 arcsec
STRAP WFS theta_beta spot size = 0.898 arcsec


In [23]:
# KAON 1322 states STRAP spot is effectively seeing-limited and has equation 16 for theta_beta
theta_beta_strap_kaon1322 = 0.5871 * (band_wvl / fried)
print(f"STRAP theta_beta using KAON 1322 equation 16 = {theta_beta_strap_kaon1322*206265:.3f} arcsec")

STRAP theta_beta using KAON 1322 equation 16 = 0.388 arcsec


#### *Sky Background and Flux*

In [4]:
# STRAP parameters
band = "R"
# Keck telescope diameter (m)
D = 10.949
# Secondary obscuration diameter (m)
Ds = 1.8
side = np.sqrt( np.pi * ( (D  / 2.0)**2 - (Ds / 2.0)**2 ) )
throughput = 0.32
ps = 1.4
time = 0.004 # 1 ms, could be 2 ms or 4 ms but try 1 ms for now
m = 14
# KAON 326
n_dark = 370 # cts/s/APD at -20 degrees C
pix_per_ap = 4
sigma_e = 0.0 # no APD readnoise, KAON 051

In [5]:
# Fixed parameters
c = 2.99792458e8   # Speed of light (m s^-1)
h = 6.6260755e-27  # Plank constant (erg s)
# Bands' names, effective wavelengths (microns), equivalent widths
# (microns), fluxes (10^-11 erg s^-1 cm^-2 A^-1) and background
# in (magnitudes arcsec^-2) [1, 2, 3].
bands = {'name': ["U", "B", "V", "R", "I", "J", "H", "K"],
         'lambd': [0.366, 0.438, 0.545, 0.641, 0.798, 1.22, 1.63, 2.19],
         'delta_lambd': [0.0665, 0.1037, 0.0909, 0.1479, 0.1042, 0.3268, 
                         0.2607, 0.5569],
         'phi_erg': [417.5, 632, 363.1, 217.7, 112.6, 31.47, 11.38, 3.961],
         'bkg_m': [21.6, 22.3, 21.1, 20.3, 19.2, 14.8, 13.4, 12.6]}
# Get band's data
band_idx = np.where(np.array(bands['name']) == band)[0][0]
# Band effective wavelength (microns)    
lambd = float(bands['lambd'][band_idx])
# Band equivalent width (microns)
delta_lamb = float(bands['delta_lambd'][band_idx])
# Flux (erg s^-1 cm^-2 A^-1)
phi_erg = float(bands['phi_erg'][band_idx])
# Background magnitude (arcsec^-2)    
bkg_m = float(bands['bkg_m'][band_idx])
# Band frequency (Hz)
f = c / (lambd * 1e-6)
# Numeric flux (s^-1 cm^-2 A^-1)
phi_n = (phi_erg * 1e-11) / ( h * f )
# Zeropoint (m = 0) number of photons on detector
n_ph_0 = phi_n * ( (side * 1e2) ** 2) * time * delta_lamb * 1e4 * throughput
# Number of star photons
n_ph_star = n_ph_0 * ( 10**(-0.4 * m ))
print(f"Number of star photons on STRAP at integration time = {n_ph_star:.3f} photons per subaperture") 
# Number of background photons (px^-1)
n_ph_bkg = n_ph_0 * ( 10**(-0.4 * bkg_m) ) * (ps)**2.0
print(f"Number of background photons on STRAP at integration time = {n_ph_bkg:.3f} background photons per pixel")

Number of star photons on STRAP at integration time = 3060.252 photons per subaperture
Number of background photons on STRAP at integration time = 18.114 background photons per pixel


In [6]:
# KAON 422 has equation for absolute number of counts per second per APD
mR = 14
counts_persecperAPD = 10**( (-mR + 26.0) / 2.5)
print(counts_persecperAPD)
counts_perAPDperframe = counts_persecperAPD*time
print(f"STRAP cts/APD at integration time {time} = {counts_perAPDperframe} cts/APD --> {counts_perAPDperframe} e-/APD")

63095.7344480193
STRAP cts/APD at integration time 0.004 = 252.3829377920772 cts/APD --> 252.3829377920772 e-/APD


In [27]:
n_dark *= time
print(n_dark)
SNR = n_ph_star / np.sqrt(n_ph_star + pix_per_ap*n_ph_bkg + n_dark*0.3 + pix_per_ap*sigma_e**2)
print(SNR)

0.37
27.336093715207316


In [28]:
# Compare to KAON 113's equation using keck_nea_photon outputs

# Zeropoint (m = 0) number of photons on detector
n_ph_0 = phi_n * ( (side * 1e2) ** 2) * delta_lamb * 1e4 * throughput # per second 
# Number of star photons
n_ph_star = n_ph_0 * ( 10**(-0.4 * m) ) 
# Number of background photons
n_ph_bkg = n_ph_0 * ( 10**(-0.4 * bkg_m) ) * ps**2.0 # per second
sigma_e = 0.1 # arbitrarily selected, thus far I have only seen no readnoise quoted for STRAP below a certain QE
# Dark current, taken from KAON because currently not calculated in keck_nea_photons (KAON 326) (see n_dark in cell above)

SNR_KAON113(n_ph_star, time, pix_per_ap, n_dark, n_ph_bkg, sigma_e)

27.57752880882451

In [29]:
# Compare to KAON 113's equation using KAON info

# KAON 422 has equation for absolute number of counts per second per APD
mR = 14
counts_persecperAPD = 10**( (-mR + 26.0) / 2.5)

# Number of background photons
n_ph_bkg = 887.6 # cts/s/APD from KAON 326 Table 4

SNR_KAON113(counts_persecperAPD, time, pix_per_ap, n_dark, n_ph_bkg, sigma_e)

7.8854382604191855

#### *Zeropoint number of photons*

In [30]:
# KAON 422 has equation for absolute number of counts per second per APD
mR = 14
counts_persecperAPD = 10**( (-mR + 26.0) / 2.5)
print(f"Absolute counts per second per APD for STRAP = {counts_persecperAPD:.3f} cts/sec/APD. At integration time = {counts_persecperAPD*time:.3f}")

Absolute counts per second per APD for STRAP = 63095.734 cts/sec/APD. At integration time = 63.096


#### *Use equations A3 and A4 from https://iopscience.iop.org/article/10.1086/316120/pdf*

In [31]:
# Fixed parameters
c = 2.99792458e8   # Speed of light (m s^-1)
h = 6.6260755e-27  # Plank constant (erg s)
# Bands' names, effective wavelengths (microns), equivalent widths
# (microns), fluxes (10^-11 erg s^-1 cm^-2 A^-1) and background
# in (magnitudes arcsec^-2) [1, 2, 3].
bands = {'name': ["U", "B", "V", "R", "I", "J", "H", "K"],
         'lambd': [0.366, 0.438, 0.545, 0.641, 0.798, 1.22, 1.63, 2.19],
         'delta_lambd': [0.0665, 0.1037, 0.0909, 0.1479, 0.1042, 0.3268, 
                         0.2607, 0.5569],
         'phi_erg': [417.5, 632, 363.1, 217.7, 112.6, 31.47, 11.38, 3.961],
         'bkg_m': [21.6, 22.3, 21.1, 20.3, 19.2, 14.8, 13.4, 12.6]}
# Get band's data
band_idx = np.where(np.array(bands['name']) == band)[0][0]
# Band effective wavelength (microns)    
lambd = float(bands['lambd'][band_idx])
# Band equivalent width (microns)
delta_lamb = float(bands['delta_lambd'][band_idx])
# Flux (erg s^-1 cm^-2 A^-1)
phi_erg = float(bands['phi_erg'][band_idx])
# Background magnitude (arcsec^-2)    
bkg_m = float(bands['bkg_m'][band_idx])
# Band frequency (Hz)
f = c / (lambd * 1e-6)
# Numeric flux (s^-1 cm^-2 A^-1)
phi_n = (phi_erg * 1e-11) / ( h * f )
# Zeropoint (m = 0) number of photons on detector
n_ph_0 = phi_n * ( (side * 1e2) ** 2) * time * delta_lamb * 1e4 * throughput

In [32]:
# Np for NGS (STRAP)
D = 10.949
Ds = 1.8
side = np.sqrt( np.pi * ( (D  / 2.0)**2 - (Ds / 2.0)**2 ) )
z = phi_n * delta_lamb * 1e4 # intensity of a zero-magnitude guide star at top of atmosphere
m = 14.0 # guide star mag
airmass = 1.35
zenith_angle = np.arccos(1/1.35) # air mass of 1.35 from header of science image that corresponds to 2024-08-07 new RTC telemetry
Ma = 0.15 # atmospheric attenuation in magnitudes per air mass
tau = 0.32 # end-to-end efficiency of optics + WFS detectors (== throughput)
t = 0.004 # integration time
A = (side * 1.0e2)**2.0 # area of the WFS subaperture in the telescope aperture plane
Np = z * 10**(-m/2.5) * tau * t * A * 10**((-1.0*(airmass - 1))*(Ma/2.5))
print(f"N_photons at integration time {t*1000.0} ms from Ellerbroek & Tyler = {Np} photons/subperature/frame")

N_photons at integration time 4.0 ms from Ellerbroek & Tyler = 2915.796090159964 photons/subperature/frame


### **LGSWFS**

Fast SHWFS for LGS

Spot size is harded coded as $\theta_{\beta} = 1.93''$ since LGS spot size on SHWFS is known. We can use KAON 1433 data to compare calculated SNR with measured SNR.

In [33]:
# LGSWFS parameters
band = "R" # not actually at V-band
side = 0.563
ps = 3.0
sigma_e = 3.6
theta_beta = 1.93 * ( np.pi/180.0 ) / ( 60.0*60.0 )
throughput = 0.36
pix_per_ap = 4
# Use frame rate --> integration time of CCD39 entries in Table 3 of KAON 1433 to compare against the measured
# SNR at those frame rates (500 Hz and 1000 Hz)
time = (1.0/500.0)
# Use magnitude of star used in KAON 1433 measurements
m = 8.1

In [34]:
# Calculate number of photons and background photons
Np, Nb = mu.n_photons(side, time, m, band, ps, throughput)
SNR = Np / np.sqrt(Np + pix_per_ap*Nb + pix_per_ap*sigma_e**2)
print(f"LGSWFS SNR at 500 Hz = {SNR}")
print(Np, Nb)
print(pix_per_ap*Nb)

LGSWFS SNR at 500 Hz = 36.24781765261076
1364.4475180464299 0.16188229215641411
0.6475291686256565


In [35]:
# LGSWFS parameters
band = "R" # not actually at V-band
side = 0.563
ps = 3.0
sigma_e = 3.6
theta_beta = 1.93 * ( np.pi/180.0 ) / ( 60.0*60.0 )
throughput = 0.36
pix_per_ap = 4
# Use frame rate --> integration time of CCD39 entries in Table 3 of KAON 1433 to compare against the measured
# SNR at those frame rates (500 Hz and 1000 Hz)
time = (1.0/1000.0)
# Use magnitude of star used in KAON 1433 measurements
m = 9.2

Np, Nb = mu.n_photons(side, time, m, band, ps, throughput)
SNR = Np / np.sqrt(Np + pix_per_ap*Nb + pix_per_ap*sigma_e**2)
print(f"LGSWFS SNR at 1000 Hz = {SNR}")

LGSWFS SNR at 1000 Hz = 14.30423055035109


These numbers agree roughly with those in KAON 1433 Table 3: LGSWFS at 500 Hz had SNR of 27 and at 1000 Hz SNR of 17.5. 

In [36]:
# LGSWFS parameters
band = "R" # not actually at V-band
side = 0.563
ps = 3.0
sigma_e = 6.5
theta_beta = 1.93 * ( np.pi/180.0 ) / ( 60.0*60.0 )
throughput = 0.36
pix_per_ap = 4*4
# Use frame rate --> integration time of CCD39 entries in Table 3 of KAON 1433 to compare against the measured
# SNR at those frame rates (500 Hz and 1000 Hz)
time = (1.0/500.0)
# Use magnitude of star used in KAON 1433 Figure 1 (same frame rates as Table 3 presented)
m = 6.5

Np, Nb = mu.n_photons(side, time, m, band, ps, throughput)
SNR = Np / np.sqrt(Np + pix_per_ap*Nb + pix_per_ap*sigma_e**2)
print(f"LGSWFS SNR at 500 Hz for R~6.5 = {SNR}")

LGSWFS SNR at 500 Hz for R~6.5 = 73.12212828545229


In [37]:
# LGSWFS parameters
band = "R" # not actually at V-band
side = 0.563
ps = 3.0
sigma_e = 6.5
theta_beta = 1.93 * ( np.pi/180.0 ) / ( 60.0*60.0 )
throughput = 0.36
pix_per_ap = 4*4
# Use frame rate --> integration time of CCD39 entries in Table 3 of KAON 1433 to compare against the measured
# SNR at those frame rates (500 Hz and 1000 Hz)
time = (1.0/1000.0)
# Use magnitude of star used in KAON 1433 Figure 1 (same frame rates as Table 3 presented)
m = 6.5

Np, Nb = mu.n_photons(side, time, m, band, ps, throughput)
SNR = Np / np.sqrt(Np + pix_per_ap*Nb + pix_per_ap*sigma_e**2)
print(f"LGSWFS SNR at 1000 Hz for R~6.5 = {SNR}")

LGSWFS SNR at 1000 Hz for R~6.5 = 49.256635741071086


These SNR values are a bit lower than those in KAON 1433 Figure 1, but those plots are estimates as well. I think these numbers are much better

#### *Ellerbroek & Tyler formulation test for laser guide star (see STRAP section for link to paper)*

In [38]:
# Fixed parameters
c = 2.99792458e8   # Speed of light (m s^-1)
h = 6.6260755e-27  # Plank constant (erg s)
# Bands' names, effective wavelengths (microns), equivalent widths
# (microns), fluxes (10^-11 erg s^-1 cm^-2 A^-1) and background
# in (magnitudes arcsec^-2) [1, 2, 3].
bands = {'name': ["U", "B", "V", "R", "I", "J", "H", "K"],
         'lambd': [0.366, 0.438, 0.545, 0.641, 0.798, 1.22, 1.63, 2.19],
         'delta_lambd': [0.0665, 0.1037, 0.0909, 0.1479, 0.1042, 0.3268, 
                         0.2607, 0.5569],
         'phi_erg': [417.5, 632, 363.1, 217.7, 112.6, 31.47, 11.38, 3.961],
         'bkg_m': [21.6, 22.3, 21.1, 20.3, 19.2, 14.8, 13.4, 12.6]}
# Get band's data
band_idx = np.where(np.array(bands['name']) == band)[0][0]
# Band effective wavelength (microns)    
lambd = float(bands['lambd'][band_idx])
# Band equivalent width (microns)
delta_lamb = float(bands['delta_lambd'][band_idx])
# Flux (erg s^-1 cm^-2 A^-1)
phi_erg = float(bands['phi_erg'][band_idx])
# Background magnitude (arcsec^-2)    
bkg_m = float(bands['bkg_m'][band_idx])
# Band frequency (Hz)
f = c / (lambd * 1e-6)
# Numeric flux (s^-1 cm^-2 A^-1)
phi_n = (phi_erg * 1e-11) / ( h * f )
# Zeropoint (m = 0) number of photons on detector
n_ph_0 = phi_n * ( (side * 1e2) ** 2) * time * delta_lamb * 1e4 * throughput

In [39]:
# Np for LGS (LGSWFS)
side = 0.563 # meters
A = (side * 1.0e2)**2.0 # area of the WFS subaperture in the telescope aperture plane
z = phi_n * delta_lamb * 1e4 # intensity of a zero-magnitude guide star at top of atmosphere
m = 8.1 # guide star mag
airmass = 1.35
Ma = 0.15 # atmospheric attenuation in magnitudes per air mass
tau = 0.36 # end-to-end efficiency of optics + WFS detectors (== throughput)
t = 0.001 # integration time
Np = z * 10**(-m/2.5) * tau * t * A * 10**((-1.0*(airmass - 1))*(Ma/2.5))
print(f"N_photons at integration time {t*1000.0} ms from Ellerbroek & Tyler = {Np} photons/subperature/frame")

N_photons at integration time 1.0 ms from Ellerbroek & Tyler = 650.0201806031159 photons/subperature/frame


In [61]:
def Idark(A, T, Eg, k):
    return A*T**(3.0/2.0)*np.exp(-Eg / (2.0*k*T)) # https://www2.keck.hawaii.edu/optics/aochar/spieart.pdf
# def Qd(T, Qd0):
#     return 122*(T**3.0)*np.exp(-6400.0/T)*Qd0

In [62]:
# 267.15 is -6 degrees Celsius -> Kelvin (-6 degrees C comes from Peltier cooled CCD, https://www2.keck.hawaii.edu/optics/aochar/spieart.pdf)
k = 8.617333262e-5 # eV ⋅ K^−1
CCD_dark_current = Idark(2.15e8, 267.15, 1.2, k)

# CCD_dark_current = Qd(267.15, 109243)

In [64]:
print(CCD_dark_current)

4.503871293784239
